In [1]:
from db import Session
from models import Product, Manufacturer, Country
session = Session()

# Exercises

Ready to create some queries on our own? Write queries that generate:

1. Products that were made in UK or USA.

In [ ]:
from sqlalchemy import select, or_
q = (select(
    Product).join(Product.countries)
    .where(or_(Country.name == 'UK', Country.name == 'USA'))
    .distinct()
)

print(q)

session.execute(q).all()

SELECT DISTINCT products.id, products.name, products.manufacturer_id, products.year, products.cpu 
FROM products JOIN products_countries AS products_countries_1 ON products.id = products_countries_1.product_id JOIN countries ON countries.id = products_countries_1.country_id 
WHERE countries.name = :name_1 OR countries.name = :name_2


[(Product(74, "Honeywell 316", "Manufacturer(33, "Honeywell")", 1969, [Country(3, "USA")], "DDP 16 Minicomputer"),),
 (Product(16, "Apple II", "Manufacturer(5, "Apple Computer")", 1977, [Country(3, "USA")], "6502"),),
 (Product(35, "Bally Astrocade", "Manufacturer(10, "Bally Consumer Products")", 1977, [Country(3, "USA")], "Z80"),),
 (Product(39, "PET", "Manufacturer(14, "Commodore")", 1977, [Country(3, "USA")], "6502"),),
 (Product(78, "Compucolor II", "Manufacturer(36, "Intelligent Systems Corporation")", 1977, [Country(3, "USA")], "8080"),),
 (Product(101, "TRS-80 Model I", "Manufacturer(52, "Radio Shack")", 1977, [Country(3, "USA")], "Z80"),),
 (Product(146, "VideoBrain Family Computer", "Manufacturer(73, "Videobrain")", 1977, [Country(3, "USA")], "Fairchild F8"),),
 (Product(65, "Exidy Sorcerer", "Manufacturer(26, "Exidy")", 1978, [Country(3, "USA")], "Z80"),),
 (Product(15, "Imagination Machine", "Manufacturer(4, "APF Electronics, Inc.")", 1979, [Country(3, "USA")], "6800"),),
 (

2. Products not made in UK or USA. Products that were made in UK and/or USA jointly with other contries should be included in the query results.

In [6]:
from sqlalchemy import select, or_, func
q = (select(
    Product).join(Product.countries)
    .where(~or_(Country.name == 'UK', Country.name == 'USA'))
    .distinct()
)

print(q)

session.execute(q).all()

SELECT DISTINCT products.id, products.name, products.manufacturer_id, products.year, products.cpu 
FROM products JOIN products_countries AS products_countries_1 ON products.id = products_countries_1.product_id JOIN countries ON countries.id = products_countries_1.country_id 
WHERE NOT (countries.name = :name_1 OR countries.name = :name_2)


[(Product(14, "CEC-I Zhonghua", "Manufacturer(3, "Tsinghua University")", 1986, [Country(2, "China")], "6502"),),
 (Product(81, "Ivel Ultra", "Manufacturer(39, "Ivasim")", 1984, [Country(13, "Croatia")], "6502 compatible"),),
 (Product(82, "Ivel Z3", "Manufacturer(39, "Ivasim")", 1983, [Country(13, "Croatia")], "6502 compatible"),),
 (Product(108, "Galeb", "Manufacturer(55, "PEL Varaždin")", 1981, [Country(13, "Croatia")], "6502"),),
 (Product(109, "Orao", "Manufacturer(55, "PEL Varaždin")", 1984, [Country(13, "Croatia")], "6502"),),
 (Product(90, "MicroBee", "Manufacturer(45, "Microbee Systems")", 1983, [Country(16, "Australia")], "Z80"),),
 (Product(94, "Microprofessor III", "Manufacturer(49, "Multitech")", 1983, [Country(17, "Taiwan")], "6502"),),
 (Product(63, "Pecom 32", "Manufacturer(25, "Elektronska Industrija Niš")", 1985, [Country(10, "Serbia")], "RCA 1802"),),
 (Product(64, "Pecom 64", "Manufacturer(25, "Elektronska Industrija Niš")", 1985, [Country(10, "Serbia")], "RCA 1802"


  Why Your Solution Works

  The key insight is: the JOIN creates one row per product-country
  combination, not one row per product.

  For a product like "Timex Sinclair 1000" with countries [UK, USA,
  Portugal], the JOIN produces 3 rows:
  ┌─────────────────────┬──────────┐
  │     Product Row     │ Country  │
  ├─────────────────────┼──────────┤
  │ Timex Sinclair 1000 │ UK       │
  ├─────────────────────┼──────────┤
  │ Timex Sinclair 1000 │ USA      │
  ├─────────────────────┼──────────┤
  │ Timex Sinclair 1000 │ Portugal │
  └─────────────────────┴──────────┘
  Your WHERE clause NOT (countries.name = 'UK' OR countries.name = 'USA')
  filters these rows, keeping only:
  ┌─────────────────────┬──────────┐
  │     Product Row     │ Country  │
  ├─────────────────────┼──────────┤
  │ Timex Sinclair 1000 │ Portugal │
  └─────────────────────┴──────────┘
  Then DISTINCT collapses this to one Product row.

  So the query effectively finds: "Products that have at least one country
  that is NOT UK/USA" - which is exactly what the exercise asks for.

  Checking List Length in SQLAlchemy

  Yes, you can test relationship counts. Here are a few approaches:

  from sqlalchemy import func, select

  # Count countries per product using a subquery
  subq = (
      select(func.count())
      .where(Product.id == products_countries.c.product_id)
      .correlate(Product)
      .scalar_subquery()
  )

  # Products with more than 1 country
  q = select(Product).where(subq > 1)

  # Or using a grouped join:
  q = (
      select(Product)
      .join(Product.countries)
      .group_by(Product.id)
      .having(func.count(Country.id) > 1)
  )

  But for your exercise, you don't need list length checks - the JOIN
  behavior naturally handles the "jointly with other countries"
  requirement.

  Does this clarify why the results include products with multiple
  countries?

In [8]:
uk_only = (
      select(Product)
      .join(Product.countries)
      .group_by(Product.id)
      .having(func.count(Country.id) == 1)  # exactly 1 country
      .having(func.max(Country.name) == 'UK')  # and it's UK
  )



In [10]:
print("UK-only products:")
uk_only_products = session.execute(uk_only).scalars().all()
for p in uk_only_products:
    print(f"  {p.name}: {p.countries}")

UK-only products:
  Acorn Atom: [Country(1, "UK")]
  BBC Micro: [Country(1, "UK")]
  Electron: [Country(1, "UK")]
  BBC Master: [Country(1, "UK")]
  Acorn Archimedes: [Country(1, "UK")]
  A7000: [Country(1, "UK")]
  CPC 464: [Country(1, "UK")]
  CPC 664: [Country(1, "UK")]
  CPC 6128: [Country(1, "UK")]
  464 Plus: [Country(1, "UK")]
  6128 Plus: [Country(1, "UK")]
  PCW: [Country(1, "UK")]
  PC-1512: [Country(1, "UK")]
  Apricot F1: [Country(1, "UK")]
  Lynx: [Country(1, "UK")]
  Dragon 32: [Country(1, "UK")]
  Dragon 64: [Country(1, "UK")]
  Enterprise 64: [Country(1, "UK")]
  Enterprise 128: [Country(1, "UK")]
  Grundy NewBrain: [Country(1, "UK")]
  Jupiter ACE: [Country(1, "UK")]
  MTX500: [Country(1, "UK")]
  MTX512: [Country(1, "UK")]
  RS128: [Country(1, "UK")]
  SAM Coupé: [Country(1, "UK")]
  Oric 1: [Country(1, "UK")]
  Oric Atmos: [Country(1, "UK")]
  Oric Telestrat: [Country(1, "UK")]
  ZX80: [Country(1, "UK")]
  ZX81: [Country(1, "UK")]
  ZX Spectrum: [Country(1, "UK")]
  

In [11]:
print("\nThese should NOT appear in exercise 2 results:")
ex2_ids = [p.id for (p,) in session.execute(q).all()]
for p in uk_only_products:
    status = "FOUND (bug!)" if p.id in ex2_ids else "not found (correct)"
    print(f"  {p.name}: {status}")


These should NOT appear in exercise 2 results:
  Acorn Atom: not found (correct)
  BBC Micro: not found (correct)
  Electron: not found (correct)
  BBC Master: not found (correct)
  Acorn Archimedes: not found (correct)
  A7000: not found (correct)
  CPC 464: not found (correct)
  CPC 664: not found (correct)
  CPC 6128: not found (correct)
  464 Plus: not found (correct)
  6128 Plus: not found (correct)
  PCW: not found (correct)
  PC-1512: not found (correct)
  Apricot F1: not found (correct)
  Lynx: not found (correct)
  Dragon 32: not found (correct)
  Dragon 64: not found (correct)
  Enterprise 64: not found (correct)
  Enterprise 128: not found (correct)
  Grundy NewBrain: not found (correct)
  Jupiter ACE: not found (correct)
  MTX500: not found (correct)
  MTX512: not found (correct)
  RS128: not found (correct)
  SAM Coupé: not found (correct)
  Oric 1: not found (correct)
  Oric Atmos: not found (correct)
  Oric Telestrat: not found (correct)
  ZX80: not found (correct)
  ZX

3. Countries with products based on the Z80 or any of its clones.

In [14]:
q = (select(
    Country).join(Product.countries)
    .where(Product.cpu.ilike("%Z80%"))
    .distinct()
)

print(q)
session.execute(q).all()

SELECT DISTINCT countries.id, countries.name 
FROM products JOIN products_countries AS products_countries_1 ON products.id = products_countries_1.product_id JOIN countries ON countries.id = products_countries_1.country_id 
WHERE lower(products.cpu) LIKE lower(:cpu_1)


[(Country(16, "Australia"),),
 (Country(4, "Netherlands"),),
 (Country(3, "USA"),),
 (Country(12, "Brazil"),),
 (Country(24, "Hungary"),),
 (Country(5, "Romania"),),
 (Country(1, "UK"),),
 (Country(23, "Poland"),),
 (Country(21, "East Germany"),),
 (Country(9, "USSR"),),
 (Country(6, "Hong Kong"),),
 (Country(11, "Japan"),),
 (Country(25, "Norway"),),
 (Country(14, "Sweden"),),
 (Country(8, "Czechoslovakia"),),
 (Country(7, "Belgium"),),
 (Country(22, "Portugal"),)]

In [15]:
q = (select(Country)
      .join(Country.products)  # relationship from Country → Product
      .where(Product.cpu.ilike("%Z80%"))
      .distinct()
  )

session.execute(q).all()

[(Country(16, "Australia"),),
 (Country(4, "Netherlands"),),
 (Country(3, "USA"),),
 (Country(12, "Brazil"),),
 (Country(24, "Hungary"),),
 (Country(5, "Romania"),),
 (Country(1, "UK"),),
 (Country(23, "Poland"),),
 (Country(21, "East Germany"),),
 (Country(9, "USSR"),),
 (Country(6, "Hong Kong"),),
 (Country(11, "Japan"),),
 (Country(25, "Norway"),),
 (Country(14, "Sweden"),),
 (Country(8, "Czechoslovakia"),),
 (Country(7, "Belgium"),),
 (Country(22, "Portugal"),)]

4. Countries that had products made in the 1970s in alphabetical order.

In [16]:
q = (select(Country)
      .join(Country.products)  # relationship from Country → Product
      .where(Product.year.between(1970,1979))
      .order_by(Country.name)
      .distinct()
  )

session.execute(q).all()

[(Country(11, "Japan"),), (Country(14, "Sweden"),), (Country(3, "USA"),)]

5. The 5 countries with the most products. If there is a tie, the query should pick the countries in alphabetical order.

In [32]:
from sqlalchemy import select, or_, func
q = (select(Country,func.count(Product.id))
    .join(Country.products)
    .group_by(Country)
    .order_by(func.count(Product.id).desc(), Country.name)
    .limit(5)
)

print(q)
session.execute(q).all()

SELECT countries.id, countries.name, count(products.id) AS count_1 
FROM countries JOIN products_countries AS products_countries_1 ON countries.id = products_countries_1.country_id JOIN products ON products.id = products_countries_1.product_id GROUP BY countries.id, countries.name ORDER BY count(products.id) DESC, countries.name
 LIMIT :param_1


[(Country(3, "USA"), 51),
 (Country(1, "UK"), 36),
 (Country(11, "Japan"), 12),
 (Country(6, "Hong Kong"), 6),
 (Country(22, "Portugal"), 6)]

In [33]:
from sqlalchemy import select, or_, func

product_count = func.count(Product.id).label('product_count')
q = (select(Country,product_count)
    .join(Country.products)
    .group_by(Country)
    .order_by(product_count.desc(), Country.name)
    .limit(5)
)

print(q)
session.execute(q).all()

SELECT countries.id, countries.name, count(products.id) AS product_count 
FROM countries JOIN products_countries AS products_countries_1 ON countries.id = products_countries_1.country_id JOIN products ON products.id = products_countries_1.product_id GROUP BY countries.id, countries.name ORDER BY product_count DESC, countries.name
 LIMIT :param_1


[(Country(3, "USA"), 51),
 (Country(1, "UK"), 36),
 (Country(11, "Japan"), 12),
 (Country(6, "Hong Kong"), 6),
 (Country(22, "Portugal"), 6)]

6. Manufacturers that have more than 3 products in UK or USA.

In [40]:
from sqlalchemy import select, or_, func

product_count = func.count(Product.id).label('product_count')
q = (select(Manufacturer, product_count)
    .join(Manufacturer.products)
    .join(Product.countries)
    .where(or_(Country.name == 'UK', Country.name == 'USA'))
    .group_by(Manufacturer)
    .having(product_count > 3)
    .distinct()
)

print(q)

session.execute(q).all()

SELECT DISTINCT manufacturers.id, manufacturers.name, count(products.id) AS product_count 
FROM manufacturers JOIN products ON manufacturers.id = products.manufacturer_id JOIN products_countries AS products_countries_1 ON products.id = products_countries_1.product_id JOIN countries ON countries.id = products_countries_1.country_id 
WHERE countries.name = :name_1 OR countries.name = :name_2 GROUP BY manufacturers.id, manufacturers.name 
HAVING count(products.id) > :param_1


[(Manufacturer(1, "Acorn Computers Ltd"), 6),
 (Manufacturer(2, "Amstrad"), 7),
 (Manufacturer(5, "Apple Computer"), 6),
 (Manufacturer(8, "Atari, Inc."), 7),
 (Manufacturer(14, "Commodore"), 10),
 (Manufacturer(52, "Radio Shack"), 6),
 (Manufacturer(63, "Sinclair Research"), 4),
 (Manufacturer(70, "Timex Sinclair"), 8)]

7. Manufacturers that have products in more than one country

In [45]:
from sqlalchemy import select, or_, func

country_count = func.count(Country.id).label('country_count')
q = (select(Manufacturer, country_count)
    .join(Manufacturer.products)
    .join(Product.countries)
    .group_by(Manufacturer)
    .having(country_count > 1)
    .order_by(country_count.desc())
    .distinct()
)


print(q)

session.execute(q).all()

SELECT DISTINCT manufacturers.id, manufacturers.name, count(countries.id) AS country_count 
FROM manufacturers JOIN products ON manufacturers.id = products.manufacturer_id JOIN products_countries AS products_countries_1 ON products.id = products_countries_1.product_id JOIN countries ON countries.id = products_countries_1.country_id GROUP BY manufacturers.id, manufacturers.name 
HAVING count(countries.id) > :param_1 ORDER BY country_count DESC


[(Manufacturer(70, "Timex Sinclair"), 15),
 (Manufacturer(14, "Commodore"), 10),
 (Manufacturer(2, "Amstrad"), 7),
 (Manufacturer(8, "Atari, Inc."), 7),
 (Manufacturer(1, "Acorn Computers Ltd"), 6),
 (Manufacturer(5, "Apple Computer"), 6),
 (Manufacturer(52, "Radio Shack"), 6),
 (Manufacturer(63, "Sinclair Research"), 4),
 (Manufacturer(9, "Atari Corporation"), 3),
 (Manufacturer(20, "Didaktik"), 3),
 (Manufacturer(44, "Memotech"), 3),
 (Manufacturer(54, "Tangerine Computer Systems"), 3),
 (Manufacturer(56, "Philips"), 3),
 (Manufacturer(57, "Pravetz"), 3),
 (Manufacturer(60, "VEB Robotron"), 3),
 (Manufacturer(62, "Sharp"), 3),
 (Manufacturer(10, "Bally Consumer Products"), 2),
 (Manufacturer(18, "EACA"), 2),
 (Manufacturer(21, "Dragon Data"), 2),
 (Manufacturer(25, "Elektronska Industrija Niš"), 2),
 (Manufacturer(27, "Intelligent Software"), 2),
 (Manufacturer(30, "Fujitsu"), 2),
 (Manufacturer(34, "IBM"), 2),
 (Manufacturer(39, "Ivasim"), 2),
 (Manufacturer(50, "NEC"), 2),
 (Manufa